# Does injury make tubules unequal? Position-resolved differential variability (notebook 107)

**The question.** All our AKI analyses compare means. Patchy injury (some tubules hit, some spared) would raise the
tubule-to-tubule variance at a position, most where about half respond; graded injury would not. Notebook 96 found
the injury score graded; proliferation and transport loss cluster in neighbourhoods.

**The frozen protocol** is `docs/results/pt-injury-heterogeneity.md` (commit dea5f8e).
- Residual over-dispersion ε per specimen × position bin × gene (moment dispersion minus a lowess trend on the mean).
- R1: a program becomes heterogeneous under injury when Δε > 0 in all four IR–control pairs and beyond both balanced
  relabelings in ≥ 3 of 8 bins. R2: does ε follow p(1 − p) of responding tubules? R3: genome-wide, exploratory.

**Sections.** 1 setup; 2 dispersion per cell (cached); 3 trend shift; 4 programs and R1; 5 R2; 6 R3; 7 verdicts;
8 figures; 9 reading and run record.

## 1 · Setup and inputs

In [ ]:
import argparse
import inspect
import json
import os
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
EXPORT = results_root / 'package_pseudospace' / 'aki' / 'pt_units.h5ad'
T71 = results_root / 'pt_pathways_package_pseudospace' / 'tables'
T76 = results_root / 'pt_package_aki' / 'tables'
output = results_root / 'pt_injury_heterogeneity'
tables, figures, cache_root = output / 'tables', output / 'figures', output / 'stage_cache'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_LOGIC_VERSION = '107.heterogeneity.1'
SEED = 107
CONTROLS, IRS = ['Ctrl1A2', 'Ctrl1A4'], ['IR2A2', 'IR2A4']
SPECIMENS = CONTROLS + IRS
RELABELINGS = {'R1': (['IR2A2', 'Ctrl1A4'], ['IR2A4', 'Ctrl1A2']), 'R2': (['IR2A2', 'Ctrl1A2'], ['IR2A4', 'Ctrl1A4'])}
N_BINS, MIN_UNITS, MIN_MEAN, FRAC, MIN_BINS, SD_FLOOR, Q_RESPOND = 8, 30, .5, .3, 3, .25, .95
for path in (EXPORT, T71 / 'universe_aki_vs_control.csv', T76 / 'positive_control_markers.csv', T76 / 'programs_transporters.csv',
             T76 / 'gene_sets.json', T76 / 'universe.csv'):
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.stats import spearmanr
from IPython.display import display

from pseudospace import heterogeneity as ht
from pseudospace.package_inputs import read_pt_units, support_positions
from pseudospace.stage_cache import cached_payload
from pseudospace.tissue_location import program_score

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
SPEC_COLOR = {'Ctrl1A2': '#0072B2', 'Ctrl1A4': '#56B4E9', 'IR2A2': '#D55E00', 'IR2A4': '#E69F00'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


guard_log = []
units = read_pt_units(EXPORT, 'aki')
obs = units.obs
mask, position = support_positions(obs.pseudospace.to_numpy(float), obs['sample'].astype(str).to_numpy(), obs.segment_class)
assert int(mask.sum()) == 14048, 'G1: common support'
GENES = pd.Index(units.var_names.astype(str))
COUNTS = sparse.csr_matrix(units.X[mask])
U = pd.DataFrame({'specimen': obs['sample'].astype(str).to_numpy()[mask], 'x': position,
                  'n_spots': obs.n_spots.to_numpy(float)[mask], 'library': np.asarray(COUNTS.sum(axis=1)).ravel()},
                 index=obs.index[mask].astype(str))
del units
U['bin'] = np.minimum((U.x * N_BINS).astype(int), N_BINS - 1)
UNIVERSE = pd.Index(pd.read_csv(T71 / 'universe_aki_vs_control.csv').gene.astype(str)).intersection(GENES)
UCOLS = GENES.get_indexer(UNIVERSE)
lo, hi = np.quantile(U.n_spots, [.25, .75])
U['size iqr'] = U.n_spots.between(lo, hi)
guard_log.append(f'G1: 14,048 common-support units; universe {len(UNIVERSE):,} genes; n_spots IQR {lo:.0f}–{hi:.0f}')

markers = pd.read_csv(T76 / 'positive_control_markers.csv', index_col=0).index.astype(str).tolist()
sets76 = json.loads((T76 / 'gene_sets.json').read_text())
u76 = set(pd.read_csv(T76 / 'universe.csv').gene.astype(str))
cyc = sorted((set(sets76['MSigDB_Hallmark_2020::G2-M Checkpoint']) | set(sets76['MSigDB_Hallmark_2020::E2F Targets'])) & u76)
tr = pd.read_csv(T76 / 'programs_transporters.csv')
tpt = sorted(tr.mouse[tr.in_T_PT.astype(bool)].astype(str))
PROGRAMS = {'INJ': markers, 'CYC': cyc, 'TPT': tpt}
assert (len(markers), len(cyc), len(tpt)) == (18, 297, 93), 'G2: program sizes'
display(U.groupby(['specimen', 'bin']).size().unstack())
print('\n'.join('· ' + g for g in guard_log))

## 2 · Dispersion per specimen × bin × gene (cached)

In [ ]:
def cells(arm):
    rows = []
    out = {}
    for s in SPECIMENS:
        for b in range(N_BINS):
            m = (U.specimen.eq(s) & U.bin.eq(b) & (U['size iqr'] if arm == 'size' else True)).to_numpy()
            n = int(m.sum())
            if n < MIN_UNITS:
                continue
            y = COUNTS[np.flatnonzero(m)][:, UCOLS]
            alpha, mean = ht.moment_dispersion(y, U.library.to_numpy()[m])
            alpha = np.where(mean >= MIN_MEAN, alpha, np.nan)
            eps, trend = ht.residual_dispersion(alpha, np.where(mean >= MIN_MEAN, mean, np.nan), frac=FRAC)
            ok = np.isfinite(trend)
            ref = float(np.interp(0.0, np.sort(np.log(mean[ok])), trend[ok][np.argsort(np.log(mean[ok]))])) if ok.sum() else np.nan
            out[f'{s}|{b}|eps'] = eps
            out[f'{s}|{b}|alpha'] = alpha
            out[f'{s}|{b}|mean'] = mean
            out[f'{s}|{b}|det'] = ht.detection(y)
            out[f'{s}|{b}|meta'] = np.array([n, ref])
    return out


CODE = inspect.getsource(cells) + inspect.getsource(ht)
DISP = {}
for arm in ('primary', 'size'):
    t0 = time.perf_counter()
    DISP[arm] = cached_payload(f'dispersion_{arm}', lambda: cells(arm), root=cache_root,
                               params={'logic': NOTEBOOK_LOGIC_VERSION, 'bins': N_BINS, 'min_units': MIN_UNITS,
                                       'min_mean': MIN_MEAN, 'frac': FRAC}, inputs={'export': str(EXPORT)}, code=CODE)
    print(f'{arm}: {sum(k.endswith("|meta") for k in DISP[arm])} cells ({time.perf_counter() - t0:.0f} s)')


def frame(arm, what):
    d = DISP[arm]
    cols = {(s, b): d[f'{s}|{b}|{what}'] for s in SPECIMENS for b in range(N_BINS) if f'{s}|{b}|{what}' in d}
    return pd.DataFrame(cols, index=UNIVERSE)


EPS = {arm: frame(arm, 'eps') for arm in DISP}
DET = frame('primary', 'det')
MEAN = frame('primary', 'mean')
long = pd.concat({'eps': EPS['primary'].stack([0, 1], future_stack=True), 'detection': DET.stack([0, 1], future_stack=True),
                  'mean': MEAN.stack([0, 1], future_stack=True)}, axis=1)
long.index.names = ['gene', 'specimen', 'bin']
long.reset_index().to_csv(tables / 'cells.csv.gz', index=False)
display(EPS['primary'].notna().sum().unstack().rename_axis(index='specimen', columns='bin'))

## 3 · Global variability: the trend shift

In [ ]:
TREND = pd.DataFrame({(s, b): {'units': DISP['primary'][f'{s}|{b}|meta'][0], 'trend at mean 1': DISP['primary'][f'{s}|{b}|meta'][1]}
                      for s in SPECIMENS for b in range(N_BINS) if f'{s}|{b}|meta' in DISP['primary']}).T
TREND.index.names = ['specimen', 'bin']
TREND.to_csv(tables / 'trend_shift.csv')
tw = TREND['trend at mean 1'].unstack('bin')
display(tw.round(3))
GLOBAL_SHIFT = (tw.loc[IRS].mean() - tw.loc[CONTROLS].mean())
print('IR − control trend shift per bin:', GLOBAL_SHIFT.round(3).to_dict())

## 4 · Programs and R1

In [ ]:
def group_diff(values, a, b):
    """values: Series indexed by specimen -> mean(a) − mean(b)."""
    return values.reindex(a).mean() - values.reindex(b).mean()


def r1_table(eps):
    rows = []
    for name, genes in PROGRAMS.items():
        g = eps.reindex([x for x in genes if x in eps.index])
        prog = g.mean(axis=0)                                             # (specimen, bin) -> program ε
        for b in range(N_BINS):
            v = pd.Series({s: prog.get((s, b), np.nan) for s in SPECIMENS})
            if v.isna().any():
                rows.append({'program': name, 'bin': b, 'Δε': np.nan, 'holds': False})
                continue
            delta = group_diff(v, IRS, CONTROLS)
            pairs = [v[i] - v[c] for i in IRS for c in CONTROLS]
            rel = {k: group_diff(v, a, bb) for k, (a, bb) in RELABELINGS.items()}
            holds = bool(min(pairs) > 0 and all(delta > abs(r) for r in rel.values()))
            rows.append({'program': name, 'bin': b, 'genes': int(g[[(s, b) for s in SPECIMENS]].notna().all(axis=1).sum()),
                         **{f'ε {s}': v[s] for s in SPECIMENS}, 'Δε': delta, 'min pair': min(pairs),
                         **{f'relabel {k}': r for k, r in rel.items()}, 'holds': holds})
    return pd.DataFrame(rows)


R1 = {arm: r1_table(EPS[arm]) for arm in EPS}
R1['primary'].to_csv(tables / 'r1.csv', index=False)
R1['size'].to_csv(tables / 'r1_size.csv', index=False)
display(R1['primary'].round(3))
R1_VERDICT = {arm: R1[arm].groupby('program').holds.sum().ge(MIN_BINS).to_dict() for arm in R1}
R1_BINS = {arm: R1[arm].groupby('program').holds.sum().to_dict() for arm in R1}
print('R1 bins holding:', R1_BINS, '\nheterogeneous programs:', R1_VERDICT)

## 5 · R2: does excess variance follow the responding share?

In [ ]:
expr = COUNTS.multiply(1e4 / np.maximum(U.library.to_numpy(), 1)[:, None]).tocsr()
expr.data = np.log1p(expr.data)
ref = U.specimen.isin(CONTROLS).to_numpy()
r2_rows, P_RESP = [], {}
for name, genes in PROGRAMS.items():
    cols = GENES.get_indexer([g for g in genes if g in GENES])
    score = program_score(expr[:, cols], ref, floor=SD_FLOOR)
    U[f'score {name}'] = score
    cut = U[ref].groupby('bin')[f'score {name}'].quantile(Q_RESPOND)
    low = U[ref].groupby('bin')[f'score {name}'].quantile(1 - Q_RESPOND)   # post hoc: loss programs respond downward
    for s in IRS:
        if name == 'TPT':
            p = U[U.specimen.eq(s)].groupby('bin').apply(lambda g: float((g[f'score {name}'] < low[g.name]).mean()))
        else:
            p = U[U.specimen.eq(s)].groupby('bin').apply(lambda g: float((g[f'score {name}'] > cut[g.name]).mean()))
        prog = EPS['primary'].reindex([x for x in genes if x in EPS['primary'].index]).mean(axis=0)
        change = pd.Series({b: prog.get((s, b), np.nan) - np.nanmean([prog.get((c, b), np.nan) for c in CONTROLS]) for b in range(N_BINS)})
        pq = p * (1 - p)
        both = change.notna() & pq.reindex(change.index).notna()
        rho = spearmanr(change[both], pq.reindex(change.index)[both]).statistic if both.sum() >= 4 else np.nan
        P_RESP[name, s] = p
        r2_rows.append({'program': name, 'specimen': s, 'ρ(ε change, p(1−p))': rho,
                        'responding share by bin': ', '.join(f'{v:.2f}' for v in p.to_numpy())})
R2 = pd.DataFrame(r2_rows)
R2.to_csv(tables / 'r2.csv', index=False)
display(R2.round(3))

## 6 · R3: genome-wide (exploratory)

In [ ]:
E = EPS['primary']


def per_gene(a, b):
    diffs = []
    for k in range(N_BINS):
        ca = [(s, k) for s in a if (s, k) in E.columns]
        cb = [(s, k) for s in b if (s, k) in E.columns]
        if len(ca) == len(a) and len(cb) == len(b):
            complete = E[ca + cb].notna().all(axis=1)              # all four specimens finite in this bin
            diffs.append((E[ca].mean(axis=1) - E[cb].mean(axis=1)).where(complete))
    return pd.concat(diffs, axis=1).mean(axis=1)


GROUPINGS = {'condition': (IRS, CONTROLS), **RELABELINGS}
STAT = pd.DataFrame({k: per_gene(a, b) for k, (a, b) in GROUPINGS.items()})
DETC = pd.concat([DET[[(s, k) for s in IRS if (s, k) in DET.columns]].mean(axis=1)
                  - DET[[(s, k) for s in CONTROLS if (s, k) in DET.columns]].mean(axis=1) for k in range(N_BINS)], axis=1).mean(axis=1)
counts_beyond = {}
for k in GROUPINGS:
    others = [o for o in GROUPINGS if o != k]
    beyond = STAT[k].abs() > STAT[others].abs().max(axis=1)
    counts_beyond[k] = {'up': int((beyond & (STAT[k] > 0)).sum()), 'down': int((beyond & (STAT[k] < 0)).sum()),
                        'genes': int(STAT[k].notna().sum())}
BEYOND = pd.DataFrame(counts_beyond).T
display(BEYOND)
member = {g: ','.join(p for p, gs in PROGRAMS.items() if g in gs) for g in STAT.index}
GENES_OUT = STAT.rename(columns={'condition': 'Δε condition'}).assign(**{'detection change': DETC, 'program': pd.Series(member)})
GENES_OUT['beyond both relabelings'] = STAT['condition'].abs() > STAT[list(RELABELINGS)].abs().max(axis=1)
GENES_OUT.to_csv(tables / 'genes.csv.gz')
TOP = GENES_OUT[GENES_OUT['beyond both relabelings'] & (GENES_OUT['Δε condition'] > 0)].sort_values('Δε condition', ascending=False).head(50)
display(TOP.round(3))

## 7 · Verdicts

In [ ]:
VERDICTS = pd.DataFrame([
    *[(f'R1 {p} becomes heterogeneous (≥ {MIN_BINS} of {N_BINS} bins)', f"{R1_BINS['primary'][p]} bins: {R1_VERDICT['primary'][p]} · size-restricted {R1_BINS['size'][p]} bins")
      for p in PROGRAMS],
    ('Recovery expectation: INJ not heterogeneous (notebook 96 graded)', str(not R1_VERDICT['primary']['INJ'])),
    ('Global trend shift IR − control (mean over bins)', f'{GLOBAL_SHIFT.mean():+.3f}'),
    ('R3 genes beyond both relabelings, up / down (condition; R1; R2)',
     '; '.join(f"{k} {v['up']}/{v['down']}" for k, v in counts_beyond.items())),
], columns=['reading', 'result'])
VERDICTS.to_csv(tables / 'verdicts.csv', index=False)
display(VERDICTS)

## 8 · Figures (plots only)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(110 * MM, 45 * MM), sharey=True)
for ax, s in zip(axes, ('Ctrl1A2', 'IR2A2')):
    b = 6
    a = DISP['primary'][f'{s}|{b}|alpha']
    m = DISP['primary'][f'{s}|{b}|mean']
    ok = np.isfinite(a)
    ax.scatter(np.log(m[ok]), np.log(a[ok]), s=1, color=MUTED, lw=0)
    e = DISP['primary'][f'{s}|{b}|eps']
    tr_ = np.log(a) - e
    order = np.argsort(np.log(m[ok]))
    ax.plot(np.log(m[ok])[order], tr_[ok][order], color=SPEC_COLOR[s])
    ax.set_xlabel('log mean count per unit'); ax.set_title(s)
axes[0].set_ylabel('log dispersion')
save(fig, '107a_dispersion_trend')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(150 * MM, 42 * MM), sharex=True)
mids = (np.arange(N_BINS) + .5) / N_BINS
for ax, p in zip(axes, PROGRAMS):
    t = R1['primary'][R1['primary'].program.eq(p)]
    ax.plot(mids[t.bin], t['Δε'], color=INK, marker='o', ms=2)
    for k, ls in zip(RELABELINGS, (':', '--')):
        ax.plot(mids[t.bin], t[f'relabel {k}'], color=MUTED, ls=ls)
    ax.axhline(0, color=LIGHT, lw=.5)
    ax.set_title(p); ax.set_xlabel('PT position')
axes[0].set_ylabel('Δ residual over-dispersion')
save(fig, '107b_program_dv')
plt.show()

fig, ax = plt.subplots(figsize=(65 * MM, 50 * MM))
for p, color in zip(PROGRAMS, ('#D55E00', '#009E73', '#0072B2')):
    g = GENES_OUT[GENES_OUT.program.str.contains(p, na=False)]
    ax.scatter(g['detection change'], g['Δε condition'], s=4, color=color, lw=0, label=p)
ax.axhline(0, color=LIGHT, lw=.5); ax.axvline(0, color=LIGHT, lw=.5)
ax.set_xlabel('detection change (IR − control)'); ax.set_ylabel('Δ residual over-dispersion')
ax.legend(loc='best')
save(fig, '107c_dv_detection')
plt.show()

## 9 · How the protocol was read, and the run record

In [ ]:
READING = pd.DataFrame([
    ('reading', 'The library N_u is the unit\'s total count over all export genes; dispersion is estimated on the universe genes.'),
    ('reading', 'The trend shift is the lowess trend evaluated at log mean count 0 (one count per unit), interpolated over the genes of each cell.'),
    ('reading', 'A bin is skipped for R1 when any of the four specimens lacks the cell; program ε averages its genes with finite ε in that cell.'),
    ('reading', 'R2 program scores are notebook 96\'s (log1p CP10k standardised on control units, SD floor 0.25); the responding cut is the control units\' 95th percentile in the same bin.'),
    ('reading', 'R3 averages each grouping\'s per-bin difference over bins where all four specimens have the cell.'),
], columns=['kind', 'what'])
READING.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
RUN = {'notebook': '107_pt_injury_heterogeneity', 'logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED, 'guards': guard_log,
       'r1_bins': R1_BINS, 'r1_verdict': R1_VERDICT, 'global_shift': GLOBAL_SHIFT.round(4).to_dict(),
       'r2': R2.round(4).to_dict('records'), 'r3_beyond': counts_beyond, 'top_genes': TOP.index[:30].tolist(),
       'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(RUN, indent=2, default=str))
print(json.dumps({k: RUN[k] for k in ('r1_bins', 'r1_verdict', 'r3_beyond', 'seconds')}, indent=2, default=str))